# Задание Ultra Pro

Решите задачу распознавания 20 русских писателей. Это подразумевает и больший размер базы для обучения соответственно. Ячейка для скачивания базы уже включена в ноутбук задания.


 В задании необходимо выполнить следующие пункты:

  1. Загрузить саму базу по ссылке и подговить файлы базы для обработки.
  2. Создать обучающую и проверочную выборки, обратив особое внимание на балансировку базы: количество примеров каждого класса должно быть примерно одного порядка. При этом для разбивки необходимо применить цикл. Проверочная выборка должна быть 20% от общей выборки.
  3. Подготовьте выборки для обучения и обучите сеть. Добейтесь результата точности сети не менее 95% на проверочной выборке модели Bag of Words и 75-80% - для модели Embedding.
   


## Загрузка данных

In [ ]:
import os
import sys
import zipfile
import subprocess

try:
    import gdown
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'])
    import gdown

gdown.download(
    'https://storage.yandexcloud.net/aiueducation/Content/base/l7/20writers.zip',
    '20writers.zip',
    quiet=True
)

os.makedirs('writers', exist_ok=True)

with zipfile.ZipFile('20writers.zip', 'r') as zip_ref:
    zip_ref.extractall('writers/')

print(os.listdir('writers'))


In [ ]:
import os
import re
import copy
import zipfile
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# выбор устройства: видеокарта, если она доступна, иначе процессор
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Устройство:', device)


In [ ]:
base_path = 'writers' # задание пути к папке с текстами

# создание списков
train_texts = []
test_texts = []
class_names = []

txt_paths = []

# поиск всех txt-файлов внутри папок
for root, dirs, files in os.walk(base_path):
    for file_name in files:
        if file_name.endswith('.txt'):
            txt_paths.append(os.path.join(root, file_name))

print('Найдено txt-файлов:', len(txt_paths))

# чтение каждого текстового файла
for file_path in txt_paths:
    # получение имени автора из названия файла
    author = os.path.basename(file_path).replace('.txt', '')
    class_names.append(author)
    with open(file_path, 'r', encoding='utf-8') as f: # чтение текста
        text = f.read()

    split = int(len(text) * 0.8)

    # разделение выборок
    train_texts.append(text[:split])
    test_texts.append(text[split:])

num_classes = len(class_names)

print('Количество классов:', num_classes)
print(class_names)

In [ ]:
# задание размеров параметров
VOCAB_SIZE = 5000
WIN_SIZE = 1000
WIN_HOP = 100

# собственный токенизатор для перевода текста в числовые последовательности
class SimpleTokenizer:
    def __init__(self, num_words, filters, lower=True, oov_token='unknown'):
        self.num_words = num_words
        self.filters = filters
        self.lower = lower
        self.oov_token = oov_token
        self.word_index = {oov_token: 1}
        self.idf = None

        # символы из filters заменяются пробелами
        self.translate_table = str.maketrans({char: ' ' for char in filters})

    def _prepare_text(self, text):
        if self.lower:
            text = text.lower()

        text = text.translate(self.translate_table)
        text = re.sub(r'\s+', ' ', text)
        return text.strip()

    def fit_on_texts(self, texts):
        counter = Counter()

        for text in texts:
            words = self._prepare_text(text).split()
            counter.update(words)

        # 0 — резерв под padding, 1 — неизвестное слово
        most_common_words = counter.most_common(self.num_words - 2)

        for index, (word, _) in enumerate(most_common_words, start=2):
            self.word_index[word] = index

    def texts_to_sequences(self, texts):
        sequences = []

        for text in texts:
            words = self._prepare_text(text).split()
            sequence = [
                self.word_index.get(word, 1)
                for word in words
                if self.word_index.get(word, 1) < self.num_words
            ]
            sequences.append(sequence)

        return sequences

    def sequences_to_tfidf_matrix(self, sequences, fit_idf=False):
        # матрица Bag of Words с весами TF-IDF
        n_docs = len(sequences)
        matrix = np.zeros((n_docs, self.num_words), dtype='float32')

        if fit_idf or self.idf is None:
            document_frequency = np.zeros(self.num_words, dtype='float32')

            for sequence in sequences:
                unique_tokens = set(token for token in sequence if 0 < token < self.num_words)
                for token in unique_tokens:
                    document_frequency[token] += 1

            self.idf = np.log((1 + n_docs) / (1 + document_frequency)) + 1

        for row_index, sequence in enumerate(sequences):
            counts = Counter(token for token in sequence if 0 < token < self.num_words)

            if not counts:
                continue

            total_tokens = sum(counts.values())

            for token, count in counts.items():
                tf = count / total_tokens
                matrix[row_index, token] = tf * self.idf[token]

        return matrix


filters = '!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n\xa0'
tokenizer = SimpleTokenizer(num_words=VOCAB_SIZE, filters=filters, lower=True, oov_token='unknown')

# обучение токенизатора
tokenizer.fit_on_texts(train_texts)

# преобразование текстов в числовые последовательности
train_seq = tokenizer.texts_to_sequences(train_texts)
test_seq = tokenizer.texts_to_sequences(test_texts)


# класс обработки текстовых последовательностей
class DataProcessor:
    def __init__(self, win_size, win_hop):
        self.win_size = win_size
        self.win_hop = win_hop

    # нарезка последовательности на окна
    def split_sequence(self, sequence):
        windows = []

        # формирование окон заданного размера
        for i in range(0, len(sequence) - self.win_size + 1, self.win_hop):
            windows.append(sequence[i:i + self.win_size])

        return windows

    # формирование обучающей и проверочной выборки
    def make_train_test(self, seq_list, max_train=None, max_test=None):
        x_train = []
        y_train = []
        x_test = []
        y_test = []

        # обработка каждого класса
        for class_index in range(len(seq_list)):
            # нарезка текста класса на окна
            windows = self.split_sequence(seq_list[class_index])

            # перемешивание окон
            np.random.shuffle(windows)

            # вычисление границы разделения на 80% и 20%
            split = int(len(windows) * 0.8)

            train_windows = windows[:split]
            test_windows = windows[split:]

            # ограничение числа окон для балансировки классов
            if max_train is not None:
                train_windows = train_windows[:max_train]

            if max_test is not None:
                test_windows = test_windows[:max_test]

            # добавление меток окнам
            x_train += train_windows
            y_train += [class_index] * len(train_windows)

            x_test += test_windows
            y_test += [class_index] * len(test_windows)

        return (
            np.array(x_train, dtype='int64'),
            np.array(y_train, dtype='int64'),
            np.array(x_test, dtype='int64'),
            np.array(y_test, dtype='int64')
        )


In [ ]:
np.random.seed(42)

# создание общего списка последовательностей
all_seq = []

# объединение обучающей и проверочной части каждого автора
for i in range(num_classes):
    all_seq.append(train_seq[i] + test_seq[i])

processor = DataProcessor(WIN_SIZE, WIN_HOP)

# формирование обучающей и проверочной выборки
x_train_seq, y_train, x_test_seq, y_test = processor.make_train_test(all_seq, max_train=300, max_test=75)

print('x_train_seq:', x_train_seq.shape)
print('y_train:', y_train.shape)
print('x_test_seq:', x_test_seq.shape)
print('y_test:', y_test.shape)

In [ ]:
# преобразование последовательностей в матрицы TF-IDF для модели Bag of Words
x_train_bow = tokenizer.sequences_to_tfidf_matrix(x_train_seq, fit_idf=True)
x_test_bow = tokenizer.sequences_to_tfidf_matrix(x_test_seq, fit_idf=False)

print(x_train_bow.shape)
print(x_test_bow.shape)


# функция вычисления точности
def calculate_accuracy(logits, targets):
    predictions = torch.argmax(logits, dim=1)
    return (predictions == targets).float().mean().item()


# общий цикл обучения для моделей PyTorch
def train_model(model, train_loader, val_loader, epochs=10, lr=1e-3, patience=None):
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {
        'accuracy': [],
        'val_accuracy': [],
        'loss': [],
        'val_loss': []
    }

    best_state = None
    best_val_accuracy = 0
    patience_counter = 0

    for epoch in range(epochs):
        model.train()

        train_loss_sum = 0
        train_accuracy_sum = 0
        train_batches = 0

        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            logits = model(x_batch)
            loss = criterion(logits, y_batch)

            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item()
            train_accuracy_sum += calculate_accuracy(logits, y_batch)
            train_batches += 1

        train_loss = train_loss_sum / train_batches
        train_accuracy = train_accuracy_sum / train_batches

        model.eval()

        val_loss_sum = 0
        val_accuracy_sum = 0
        val_batches = 0

        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch = x_batch.to(device)
                y_batch = y_batch.to(device)

                logits = model(x_batch)
                loss = criterion(logits, y_batch)

                val_loss_sum += loss.item()
                val_accuracy_sum += calculate_accuracy(logits, y_batch)
                val_batches += 1

        val_loss = val_loss_sum / val_batches
        val_accuracy = val_accuracy_sum / val_batches

        history['loss'].append(train_loss)
        history['accuracy'].append(train_accuracy)
        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(val_accuracy)

        print(
            f'Эпоха {epoch + 1}/{epochs} | '
            f'loss: {train_loss:.4f} | accuracy: {train_accuracy:.4f} | '
            f'val_loss: {val_loss:.4f} | val_accuracy: {val_accuracy:.4f}'
        )

        if patience is not None:
            if val_accuracy > best_val_accuracy:
                best_val_accuracy = val_accuracy
                best_state = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= patience:
                print('Ранняя остановка обучения')
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return history


# функция построения графиков
def plot_training_history(history, title):
    plt.figure(figsize=(5, 3))
    plt.plot(history['accuracy'], label='Обучение')
    plt.plot(history['val_accuracy'], label='Проверка')
    plt.title(f'{title} — точность')
    plt.xlabel('Эпоха')
    plt.ylabel('Точность')
    plt.legend()
    plt.grid()
    plt.show()

    plt.figure(figsize=(5, 3))
    plt.plot(history['loss'], label='Обучение')
    plt.plot(history['val_loss'], label='Проверка')
    plt.title(f'{title} — ошибка')
    plt.xlabel('Эпоха')
    plt.ylabel('Ошибка')
    plt.legend()
    plt.grid()
    plt.show()


# модель Bag of Words на PyTorch
class BagOfWordsClassifier(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)


x_train_bow_tensor = torch.tensor(x_train_bow, dtype=torch.float32)
x_test_bow_tensor = torch.tensor(x_test_bow, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_loader_bow = DataLoader(
    TensorDataset(x_train_bow_tensor, y_train_tensor),
    batch_size=32,
    shuffle=True
)

test_loader_bow = DataLoader(
    TensorDataset(x_test_bow_tensor, y_test_tensor),
    batch_size=32,
    shuffle=False
)

model_bow = BagOfWordsClassifier(VOCAB_SIZE, num_classes)

history_bow = train_model(
    model_bow,
    train_loader_bow,
    test_loader_bow,
    epochs=8,
    lr=1e-3,
    patience=3
)

plot_training_history(history_bow, 'Bag of Words')


In [ ]:
# модель Embedding + Conv1D на PyTorch
class EmbeddingConvClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_classes):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        self.conv = nn.Conv1d(
            in_channels=embedding_dim,
            out_channels=128,
            kernel_size=5
        )

        self.pool = nn.AdaptiveMaxPool1d(1)

        self.classifier = nn.Sequential(
            nn.Linear(128, 100),
            nn.ReLU(),
            nn.Linear(100, num_classes)
        )

    def forward(self, x):
        x = self.embedding(x)          # [batch, длина окна, размер embedding]
        x = x.permute(0, 2, 1)         # [batch, размер embedding, длина окна]
        x = torch.relu(self.conv(x))   # [batch, 128, новая длина]
        x = self.pool(x).squeeze(-1)   # [batch, 128]
        x = self.classifier(x)

        return x


x_train_seq_tensor = torch.tensor(x_train_seq, dtype=torch.long)
x_test_seq_tensor = torch.tensor(x_test_seq, dtype=torch.long)

train_loader_emb = DataLoader(
    TensorDataset(x_train_seq_tensor, y_train_tensor),
    batch_size=32,
    shuffle=True
)

test_loader_emb = DataLoader(
    TensorDataset(x_test_seq_tensor, y_test_tensor),
    batch_size=32,
    shuffle=False
)

model_emb = EmbeddingConvClassifier(
    vocab_size=VOCAB_SIZE,
    embedding_dim=64,
    num_classes=num_classes
)

history_emb = train_model(
    model_emb,
    train_loader_emb,
    test_loader_emb,
    epochs=10,
    lr=1e-3,
    patience=None
)

plot_training_history(history_emb, 'Embedding')
